# EDA + Experiment Setup

In [1]:
from databricks.connect import DatabricksSession
from pyspark.sql import functions
from pyspark.sql.window import Window
spark = DatabricksSession.builder.getOrCreate()

In [2]:
CATALOG = "commercial-insurance-underwriting-solution-catalog"
SEED = 42

## EDA

In [3]:
policy = spark.table(f"`{CATALOG}`.silver.policy")

- Each id tracked to see how many years it has been used - signalling renewal since new business?

In [5]:
policy_year_per_id = policy.groupBy('ID').count().withColumnRenamed('count', 'number_of_year_of_policy')
policy_year_per_id.groupBy('number_of_year_of_policy').count().withColumnRenamed('count', 'n_policy').orderBy('number_of_year_of_policy').show()

+------------------------+--------+
|number_of_year_of_policy|n_policy|
+------------------------+--------+
|                       1|   20579|
|                       2|   14685|
|                       3|   17346|
|                       4|     892|
+------------------------+--------+



- An example to trace

In [6]:
example_id = policy_year_per_id.filter('number_of_year_of_policy == 4').first()['ID']
policy.filter(functions.col("ID") == example_id).orderBy("Date_last_renewal").select(
      "ID", "Date_start_contract", "Date_last_renewal", "Date_next_renewal",
      "Premium", "Cost_claims_year", "N_claims_year", "Type_risk", "Area", "Power",
  ).show(truncate=False)

+---+-------------------+-----------------+-----------------+-------+----------------+-------------+---------+----+-----+
|ID |Date_start_contract|Date_last_renewal|Date_next_renewal|Premium|Cost_claims_year|N_claims_year|Type_risk|Area|Power|
+---+-------------------+-----------------+-----------------+-------+----------------+-------------+---------+----+-----+
|1  |2015-11-05         |2015-11-05       |2016-11-05       |222.52 |0.0             |0            |1        |0   |80   |
|1  |2015-11-05         |2016-11-05       |2017-11-05       |213.78 |0.0             |0            |1        |0   |80   |
|1  |2015-11-05         |2017-11-05       |2018-11-05       |214.84 |0.0             |0            |1        |0   |80   |
|1  |2015-11-05         |2018-11-05       |2019-11-05       |216.99 |0.0             |0            |1        |0   |80   |
+---+-------------------+-----------------+-----------------+-------+----------------+-------------+---------+----+-----+



- Does each policy since new business line up year after year before churn?

In [7]:
w = Window.partitionBy('ID').orderBy('Date_last_renewal')
chained_policies = policy.withColumn('previous_next_renewal', functions.lag('Date_next_renewal').over(w)).withColumn('row_number', functions.row_number().over(w))
chained_policies.filter(functions.col('row_number') > 1).withColumn('chains_correctly', functions.col('Date_last_renewal') ==  functions.col('previous_next_renewal')).groupBy('chains_correctly').count().show()

+----------------+-----+
|chains_correctly|count|
+----------------+-----+
|            true|52053|
+----------------+-----+



- Lets look up the key target variables distribution